# Reduce the parameter interval to the minimum interval that contains all working simulations

In [ ]:
import numpy as np

In [ ]:
basefolder = f"/data/HCM/5/scenarios/50/"
output_mask = "output_mask_beat_5.txt"

In [ ]:
mask = np.loadtxt(f"{basefolder}/output/{output_mask}",dtype=int)

input_parameters = np.loadtxt(f"{basefolder}/data/X.txt",dtype=float)

input_working = input_parameters[mask == 1]

min_values_original = np.min(input_parameters, axis=0)
max_values_original = np.max(input_parameters, axis=0)

intervals_original = np.column_stack((min_values_original, max_values_original))

min_values_working = np.min(input_working, axis=0)
max_values_working = np.max(input_working, axis=0)

intervals_working = np.column_stack((min_values_working, max_values_working))


In [ ]:

# Check if points are within the bounds
within_bounds = np.all((input_parameters >= intervals_original[:, 0]) & (input_parameters <= intervals_original[:, 1]), axis=1)

# Calculate the amount and percentage of points within bounds
amount_within_bounds = np.sum(within_bounds)
percentage_within_bounds = (np.sum(mask) / amount_within_bounds) * 100

print(f"Working: {np.sum(mask)}")

print(f"Initial amount of points: {amount_within_bounds}")
print(f"Initial success rate: {percentage_within_bounds:.2f}%")


# Check if points are within the bounds
within_bounds_working = np.all((input_parameters >= intervals_working[:, 0]) & (input_parameters <= intervals_working[:, 1]), axis=1)

# Calculate the amount and percentage of points within bounds
amount_within_bounds_working = np.sum(within_bounds_working)
percentage_within_bounds_working = (np.sum(mask) / amount_within_bounds_working) * 100

print(f"Reduced amount of points: {amount_within_bounds_working}")
print(f"Increased success rate: {percentage_within_bounds_working:.2f}%")



In [ ]:
np.set_printoptions(suppress=True)

for i in range(len(intervals_original)):
	increase_min = 100*(intervals_working[i][0] - intervals_original[i][0])/(intervals_original[i][1]-intervals_original[i][0])
	
	if increase_min > 5:
		print(f"Label {i}")
		print(f"Original interval: {intervals_original[i]}")
		print(f"Reduced interval: {intervals_working[i]}")
		print(f"Change: min increase by {increase_min}%")
	else:
		decrease_max = 100*(intervals_original[i][1] - intervals_working[i][1])/(intervals_original[i][1]-intervals_original[i][0])

		if decrease_max > 5:
			print(f"Label {i}")
			print(f"Original interval: {intervals_original[i]}")
			print(f"Reduced interval: {intervals_working[i]}")
			print(f"Change: max decreased by {decrease_max}%")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt


# Filter the rows where mask is 1
input_working = input_parameters[mask == 1]

# Function to calculate success rate
def calculate_success_rate(parameters, intervals):
    within_bounds = np.all((parameters >= intervals[:, 0]) & (parameters <= intervals[:, 1]), axis=1)
    return 100*np.sum(mask) / np.sum(within_bounds)

# Calculate initial success rate
initial_success_rate = calculate_success_rate(input_parameters, intervals_original)

# Calculate success rate increase for each dimension
success_rate_increase = []
dimensions_to_remove = []
final_success_rate = [initial_success_rate]

for i in range(intervals_original.shape[0]):
    reduced_intervals = intervals_original.copy()
    reduced_intervals[i, 0] = np.min(input_working[:, i])
    reduced_intervals[i, 1] = np.max(input_working[:, i])
    success_rate = calculate_success_rate(input_parameters, reduced_intervals)

for i in range(success_rate.shape[0]):

    success_rate_increase.append(success_rate - initial_success_rate)


dimensions_to_remove.append(np.argmax(success_rate_increase))
final_success_rate.append(np.max(success_rate_increase))

# # Sort dimensions based on success rate increase
# sorted_dimensions = np.argsort(success_rate_increase)[::-1]

# # Iteratively reduce intervals and calculate success rate
# success_rates = [initial_success_rate]
# for i in range(1, len(sorted_dimensions) + 1):
#     reduced_intervals = intervals_original.copy()
#     for j in sorted_dimensions[:i]:
#         reduced_intervals[j, 0] = np.min(input_working[:, j])
#         reduced_intervals[j, 1] = np.max(input_working[:, j])
#     success_rate = calculate_success_rate(input_parameters, reduced_intervals)
#     success_rates.append(success_rate)

# # Plot the results
# plt.figure(figsize=(10, 6))
# plt.plot(range(len(success_rates)), success_rates, marker='o')
# plt.xlabel('Number of Labels')
# plt.ylabel('Success Rate')
# plt.title('Success Rate vs Number of Labels')
# plt.grid(True)
# plt.show()


# print(success_rate)

In [ ]:
sorted_dimensions

In [ ]:
success_rate_increase